<a href="https://colab.research.google.com/github/vaniatharv/AccidentLens/blob/main/Accident_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install tqdm

In [ ]:
!pip install tensorflow

In [ ]:
!pip install opencv-python

In [ ]:
import pandas as pd

csv_path = "/content/drive/MyDrive/train.csv"
df = pd.read_csv(csv_path)
df.head()

,filename,case,duration
0,accident_000.mp4,accident,8.31
1,accident_001.mp4,accident,5.35
2,accident_0010.mp4,accident,6.71
3,accident_0011.mp4,accident,6.40
4,accident_0012.mp4,accident,7.08


In [ ]:
df['case'].values_counts()

,count
case,
accident,68
driving,68


# extract frames

In [ ]:
import cv2, os
from tqdm import tqdm

def extract_frames(video_path, output_folder, every_n_frames=10):
  os.makedirs(output_folder, exist_ok=True)
  cap = cv2.VideoCapture(video_path)
  frame_id, count = 0, 0

  while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
      break
    if frame_id % every_n_frames == 0:
      frame_path = os.path.join(output_folder, f"frame_{count}.jpg")
      cv2.imwrite(frame_path, frame)
      count += 1
    frame_id += 1

  cap.release()

base_video_dir = "/content/drive/MyDrive/train/training_videos"
base_frame_dir = "/content/drive/MyDrive/train/training_frames"

for video_file, row in tqdm(df.iterrows(), total=len(df)):
  video_file = os.path.join(base_video_dir, row['filename'])
  label = row['case']
  output_folder = os.path.join(base_frame_dir, label, row['filename'].split('.')[0])
  extract_frames(video_file, output_folder, every_n_frames =10)

100%|██████████| 136/136 [03:13<00:00,  1.42s/it]


# feature extraction

In [ ]:
!pip install keras

In [ ]:
!pip install tensorflow

In [ ]:
import numpy as np
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing import image
import glob

In [ ]:
# Load the pre-trained EfficientNetB0 model
model = EfficientNetB0(weights='imagenet', include_top=False, pooling='avg')

def extract_features_from_frame(frame_path):
    img = image.load_img(frame_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)
    features = model.predict(img_array)
    return features.flatten()

# Directory containing the extracted frames
base_frame_dir = "/content/drive/MyDrive/train/training_frames"

# Dictionary to store features
video_features = {}

# Iterate through each video folder and extract features from frames
for case_folder in os.listdir(base_frame_dir):
    case_path = os.path.join(base_frame_dir, case_folder)
    if os.path.isdir(case_path):
        for video_folder in os.listdir(case_path):
            video_path = os.path.join(case_path, video_folder)
            if os.path.isdir(video_path):
                frame_files = glob.glob(os.path.join(video_path, "*.jpg"))
                frame_features = [extract_features_from_frame(frame_file) for frame_file in tqdm(frame_files, desc=f"Extracting features for {video_folder}")]
                video_features[video_folder] = np.array(frame_features)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Extracting features for accident_000:   0%|          | 0/25 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step


Extracting features for accident_000:   4%|▍         | 1/25 [00:03<01:16,  3.20s/it]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step


...

Extracting features for driving_009:  50%|█████     | 2/4 [00:00<00:00,  4.61it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 178ms/step


Extracting features for driving_009:  75%|███████▌  | 3/4 [00:00<00:00,  3.84it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step


Extracting features for driving_009: 100%|██████████| 4/4 [00:01<00:00,  3.35it/s]


In [ ]:
# You can now inspect the extracted features
# For example, print the shape of features for one video
print(video_features['accident_000'].shape)

(25, 1280)


# dataset preparation

In [ ]:
import glob
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences
y = np.array([1 if label == 'accident' else 0 for label in df['case']])

# Extract the feature arrays from the dictionary
video_feature_list = list(video_features.values())

x = pad_sequences(video_feature_list, padding='post', dtype='float32')

x_train, x_test, y_train, y_test = train_test_split(x, y,
                                                    test_size=0.2,
                                                    stratify=y,
                                                    random_state=42)

# trainig the model


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Define the LSTM model
model = Sequential()
model.add(LSTM(128, return_sequences=True, input_shape=(x_train.shape[1], x_train.shape[2])))
model.add(Dropout(0.2))
model.add(LSTM(64))
model.add(Dropout(0.2))
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_4 (LSTM)                   │ (None, 40, 128)        │       721,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 40, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 770,881 (2.94 MB)

 Trainable params: 770,881 (2.94 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(x_train, y_train,
                    epochs=10,
                    batch_size=8,
                    validation_split=0.2)

Epoch 1/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 6s 212ms/step - accuracy: 0.5296 - loss: 0.6551 - val_accuracy: 0.8182 - val_loss: 0.6447
Epoch 2/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 2s 136ms/step - accuracy: 0.9618 - loss: 0.2888 - val_accuracy: 1.0000 - val_loss: 0.0722
Epoch 3/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 2s 160ms/step - accuracy: 0.9882 - loss: 0.1052 - val_accuracy: 0.9091 - val_loss: 0.3542
Epoch 4/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 96ms/step - accuracy: 0.9261 - loss: 0.2617 - val_accuracy: 0.9091 - val_loss: 0.1974
Epoch 5/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 108ms/step - accuracy: 0.9663 - loss: 0.1530 - val_accuracy: 0.9091 - val_loss: 0.3058
Epoch 6/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 107ms/step - accuracy: 0.9061 - loss: 0.2912 - val_accuracy: 0.9091 - val_loss: 0.2995
Epoch 7/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 101ms/step - accuracy: 0.9364 - loss: 0.2392 - val_accuracy: 0.8636 - val_loss: 0.3323
Epoch 8/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 129ms/step - accuracy: 0.9062 - loss: 0.2466 - val_accuracy: 0.8

In [ ]:
loss, accuracy = model.evaluate(x_test, y_test)
print(f"Test Accuracy: {accuracy*100:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step - accuracy: 0.9643 - loss: 0.1512
Test Accuracy: 96.43%


In [ ]:
model.save("/content/drive/MyDrive/model.h5")

# prediction


In [ ]:
import numpy as np
import cv2, os, glob
from tensorflow.keras.models import load_model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load trained LSTM model
model = load_model("/content/drive/MyDrive/model.h5")

# Load feature extractor
feature_extractor = EfficientNetB0(weights='imagenet', include_top=False, pooling='avg')

In [ ]:
def extract_frames(video_path, every_n_frames=10):
    frames = []
    cap = cv2.VideoCapture(video_path)
    frame_id = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_id % every_n_frames == 0:
            frames.append(frame)
        frame_id += 1
    cap.release()
    return frames

In [ ]:
def extract_video_features(video_path):
    frames = extract_frames(video_path, every_n_frames=10)
    features = []
    for frame in frames:
        img = cv2.resize(frame, (224, 224))
        x = image.img_to_array(img)
        x = np.expand_dims(x, axis=0)
        x = preprocess_input(x)
        feat = feature_extractor.predict(x, verbose=0)
        features.append(feat[0])
    return np.array(features)

In [ ]:
def predict_video(video_path):
    features = extract_video_features(video_path)
    X = pad_sequences([features], padding='post', dtype='float32', maxlen=model.input_shape[1])
    pred = model.predict(X)
    label = "Accident" if pred[0][0] > 0.5 else "Normal Driving"
    print(f"Prediction: {label} (Confidence: {pred[0][0]:.2f})")

In [ ]:
video_path = "/content/drive/MyDrive/train/training_videos/accident_0020.mp4"
predict_video(video_path)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 467ms/step
Prediction: Accident (Confidence: 1.00)
